In [ ]:
!pip install scikit-learn
!pip install pandas
!pip install sentencepiece

In [2]:
# filter dataset
!python3 MT-Preparation/filtering/filter.py ./en-zh.en ./en-zh.zh en zh

Dataframe shape (rows, columns): (231267, 2)
--- Rows with Empty Cells Deleted	--> Rows: 231267
--- Duplicates Deleted			--> Rows: 229646
--- Source-Copied Rows Deleted		--> Rows: 229640
--- Too Long Source/Target Deleted	--> Rows: 224743
--- HTML Removed			--> Rows: 224743
--- Rows will remain true-cased		--> Rows: 224743
--- Rows with Empty Cells Deleted	--> Rows: 224743
--- Source Saved: ./en-zh.en-filtered.en
--- Target Saved: ./en-zh.zh-filtered.zh


In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def extract_balanced_salient_words_tfidf(corpus, window_size=4, top_k=5):
    results = []
    n = len(corpus)

    for i in range(n):
        # Determine how many sentences to pull from before and after
        if i < window_size // 2:
            num_before = i
            num_after = window_size - num_before
        elif i > n - window_size // 2 - 1:
            num_after = n - i - 1
            num_before = window_size - num_after
        else:
            num_before = window_size // 2
            num_after = window_size // 2

        # Collect indices for context sentences (excluding the current one)
        before = list(range(max(0, i - num_before), i))
        after = list(range(i + 1, min(n, i + 1 + num_after)))
        context_indices = before + after
        pseudo_doc = [corpus[j] for j in context_indices]

        # If no context (corner case), append empty list
        if not pseudo_doc:
            results.append([])
            continue

        # Extract salient keywords with TF-IDF
        vectorizer = TfidfVectorizer(stop_words='english')
        tfidf = vectorizer.fit_transform(pseudo_doc)
        scores = np.asarray(tfidf.sum(axis=0)).flatten()
        feature_names = np.array(vectorizer.get_feature_names_out())

        top_indices = np.argsort(scores)[::-1][:top_k]
        salient_words = feature_names[top_indices].tolist()
        results.append(salient_words)

    return results

def write_salient_prefixed_raw_file(raw_lines, output_file, salient_word_lists):
    with open(output_file, "w") as f:
        for line, salient_words in zip(raw_lines, salient_word_lists):
            # Construct full line: salient words + separator + original sentence
            prefixed_line = ' '.join(salient_words + ['__SEP__'] + line.split())
            f.write(prefixed_line + '\n')

print("preparing")
raw_lines = open("en-zh.en-filtered.en", "r").read().splitlines()
print("extracting")
salient_contexts = extract_balanced_salient_words_tfidf(raw_lines, window_size=4, top_k=5)
print("writing")
write_salient_prefixed_raw_file(raw_lines, "en-zh.en-filtered-salient.en", salient_contexts)


preparing
extracting
writing


In [ ]:
salient_words_per_sentence = extract_balanced_salient_words_tfidf(raw_lines)

In [13]:
# train a sentencepiece model for subwording
!python3 MT-Preparation/subwording/1-train_unigram.py ./en-zh.en-filtered.en ./en-zh.zh-filtered.zh

sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=./en-zh.en-filtered.en --model_prefix=source --vocab_size=10000 --hard_vocab_limit=false --split_digits=true --user_defined_symbols=__SEP__
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ./en-zh.en-filtered.en
  input_format: 
  model_prefix: source
  model_type: UNIGRAM
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: __SEP__
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  see

In [14]:
# subword the dataset
!python3 MT-Preparation/subwording/2-subword.py source.model target.model ./en-zh.en-filtered.en ./en-zh.zh-filtered.zh

Source Model: source.model
Target Model: target.model
Source Dataset: ./en-zh.en-filtered.en
Target Dataset: ./en-zh.zh-filtered.zh
Done subwording the source file! Output: ./en-zh.en-filtered.en.subword
Done subwording the target file! Output: ./en-zh.zh-filtered.zh.subword


In [17]:
# first 3 lines before subwording
!head -n 3 ./en-zh.en-filtered.en && echo "-----" && head -n 3 ./en-zh.zh-filtered.zh

en
Thank you so much, Chris. And it's truly a great honor to have the opportunity to come to this stage twice; I'm extremely grateful.
I have been blown away by this conference, and I want to thank all of you for the many nice comments about what I had to say the other night.
-----
zh
非常谢谢，克里斯。的确非常荣幸 能有第二次站在这个台上的机会，我真是非常感激。
这个会议真是让我感到惊叹不已，我还要谢谢你们留下的 关于我上次演讲的精彩评论


In [18]:
# first 3 lines after subwording
!head -n 3 ./en-zh.en-filtered.en.subword && echo "---" && head -n 3 ./en-zh.zh-filtered.zh.subword after

▁en
▁Thank ▁you ▁so ▁much , ▁Chris . ▁And ▁it ' s ▁truly ▁a ▁great ▁honor ▁to ▁have ▁the ▁opportunity ▁to ▁come ▁to ▁this ▁stage ▁twice ; ▁I ' m ▁extremely ▁grateful .
▁I ▁have ▁been ▁blown ▁away ▁by ▁this ▁conference , ▁and ▁I ▁want ▁to ▁thank ▁all ▁of ▁you ▁for ▁the ▁many ▁nice ▁comment s ▁about ▁what ▁I ▁had ▁to ▁say ▁the ▁other ▁night .
---
==> ./en-zh.zh-filtered.zh.subword <==
▁ z h
▁非常 谢谢 , 克里斯 。 的确 非常 荣幸 ▁能 有 第二次 站在 这个 台上 的机会 , 我 真是 非常 感激 。
▁这个 会议 真是 让我 感到 惊 叹 不 已 , 我 还要 谢谢你们 留下 的 ▁关于 我 上 次 演讲 的 精彩 评论
head: cannot open 'after' for reading: No such file or directory


In [19]:
# split the dataset into training set, development set, and test set
# Development and test sets should be between 1000 and 5000 segments (here we chose 200)
!python3 MT-Preparation/train_dev_split/train_dev_test_split.py 2000 2000 ./en-zh.en-filtered.en.subword ./en-zh.zh-filtered.zh.subword

Dataframe shape: (224743, 2)
--- Empty Cells Deleted --> Rows: 224743
--- Wrote Files
Done!
Output files
./en-zh.en-filtered.en.subword.train
./en-zh.zh-filtered.zh.subword.train
./en-zh.en-filtered.en.subword.dev
./en-zh.zh-filtered.zh.subword.dev
./en-zh.en-filtered.en.subword.test
./en-zh.zh-filtered.zh.subword.test


In [9]:
!wc -l ./*.subword.*

     2000 ./en-zh.en-filtered-salient.en.subword.dev
     2000 ./en-zh.en-filtered-salient.en.subword.test
     2000 ./en-zh.en-filtered-salient.en.subword.test.desubword
   220743 ./en-zh.en-filtered-salient.en.subword.train
     2000 ./en-zh.en-filtered.en.subword.dev
     2000 ./en-zh.en-filtered.en.subword.test
     2000 ./en-zh.en-filtered.en.subword.test.desubword
   220743 ./en-zh.en-filtered.en.subword.train
      200 ./en-zh.en-filtered.en.subword.translated
      200 ./en-zh.en-filtered.en.subword.translated.desubword
     2000 ./en-zh.zh-filtered.zh.subword.dev
     2000 ./en-zh.zh-filtered.zh.subword.test
     1999 ./en-zh.zh-filtered.zh.subword.test.cleaned
     2000 ./en-zh.zh-filtered.zh.subword.test.desubword
   220743 ./en-zh.zh-filtered.zh.subword.train
   682628 total


In [20]:
# check the first and last line from each dataset
!echo "---First line---"
!head -n 1 ./*.{train,dev,test}

!echo -e "\n---Last line---"
!tail -n 1 ./*.{train,dev,test}

---First line---
==> ./en-zh.en-filtered-salient.en.subword.train <==
en

==> ./en-zh.en-filtered.en.subword.train <==
▁en

==> ./en-zh.zh-filtered.zh.subword.train <==
▁ z h

==> ./en-zh.en-filtered-salient.en.subword.dev <==
▁states ▁yes ▁racism ▁predictable ▁rural ▁ __SEP__ ▁So ▁it ' s ▁the ▁combination ▁of ▁these ▁two ▁things : ▁it ' s ▁education ▁and ▁the ▁type ▁of ▁neighbors ▁that ▁you ▁have , ▁which ▁we ' ll ▁talk ▁about ▁more ▁in ▁a ▁moment .

==> ./en-zh.en-filtered.en.subword.dev <==
▁So ▁it ' s ▁the ▁combination ▁of ▁these ▁two ▁things : ▁it ' s ▁education ▁and ▁the ▁type ▁of ▁neighbors ▁that ▁you ▁have , ▁which ▁we ' ll ▁talk ▁about ▁more ▁in ▁a ▁moment .

==> ./en-zh.zh-filtered.zh.subword.dev <==
▁所以 这两 样东西 是 联合 起来 的 。 ▁其实 就是 你的 受 教育 程度 和 周围 邻居 的 类型 , ▁我们 一会儿 再 具体 的 谈 一 谈 。

==> ./en-zh.en-filtered-salient.en.subword.test <==
▁believe ▁don ▁just ▁hasn ▁happened ▁ __SEP__ ▁D ic t ator ship s ▁in ▁C ze cho s lo va k ia , ▁East ▁Germany , ▁E s ton ia , ▁La t vi a , ▁Li t hu 